In [ ]:
import numpy as np

from common import *
from torch.func import functional_call
from torch.utils._pytree import tree_map

from opacus import PrivacyEngine

In [ ]:
# Experiment
n_epochs = 50
n_exp = 10
class_names = ["clear-day", "snow-day", "clear-evening", "snow-evening"]

# Training
lr = 0.25
momentum = 0.9
weight_decay = 1e-4
bs_train = 128
bs_test = 64
train_ratio = 0.8
schedule = False

# Privacy
private = True
privacy_config = {
    "clipping": "flat",
    "grad_sample_mode": "no_op",
    "max_grad_norm": 1.5,
    "target_delta": 1e-5,
    "target_epsilon": 10.0,
}

# Data / quantum
qpc = 10
b_size = 2 ** (qpc // 2)
dim1_blocks = 2
dim2_blocks = 3
dim1 = dim1_blocks * b_size
dim2 = dim2_blocks * b_size
input_shape = (dim1, dim2)

loader_dir = "./loaders/4class_2000/qasm"
sample_loader = f"{loader_dir}/clear-day/BAE_10q_clear-day_210000000171_21960_0_0.qasm2"

# Model-specific
classifier_layers = 3
postprocess = False
num_channels = 1
measure_wires = [0] if postprocess else list(range(len(class_names)))

configs = [
    {
        "embedding": "amplitude",
        "circuits": [num_channels],
        "qubits_per_circ": [qpc],
        "layers": [classifier_layers],
        "measure_wires": [measure_wires],
        "qubits_per_circ_ang": [20],
        "layers_ang": [12],
    },
]

In [ ]:
class VQC:
    def __init__(self, qubits, measure_wires, num_layers, loader_tape=None, loader=None):
        self.wires = range(0, qubits)
        self.num_qubits = qubits
        self.layers = num_layers
        self.measure_wires = measure_wires
        self.loader_tape = loader_tape
        if loader is not None:
            self.loader = loader

    def variational_ansatz(self, var_params):
        for i in range(self.num_qubits):
            qml.RZ(var_params[i * 3], wires=i)
            qml.RY(var_params[i * 3 + 1], wires=i)
            qml.RZ(var_params[i * 3 + 2], wires=i)

        for i in range(self.num_qubits):
            qml.CNOT(wires=[i, (i + 1) % self.num_qubits])

    def circuit_template(self, inputs, var_params):
        self.loader(inputs, self.loader_tape)
        qml.layer(self.variational_ansatz, self.layers, var_params)

    def get_circuit(self, var_params, inputs=None):
        self.circuit_template(inputs, var_params)
        return [qml.expval(qml.PauliZ(i)) for i in self.measure_wires]


class BAENet(nn.Module):
    def __init__(self, num_classes, connection_layer=nn.Identity(), qlayer_config={}, noise_model=None):
        super().__init__()

        self.embedding = qlayer_config["embedding"]
        self.param_count = 0

        circuits = qlayer_config["circuits"]
        num_layers = qlayer_config["layers"]
        measure_wires = qlayer_config["measure_wires"]

        var_params_shape = (num_layers, 3 * qpc)
        self.q_layers = nn.ModuleList()
        self.q_circuits = [[0] * dim2_blocks for i in range(dim1_blocks)]
        self.q_nodes = [[0] * dim2_blocks for i in range(dim1_blocks)]

        q_device = qml.device("default.qubit", wires=qpc)

        for i in range(dim1_blocks):
            ml = nn.ModuleList()
            for j in range(dim2_blocks):
                vqc = VQC(qpc, measure_wires, num_layers, get_loader_circuit_tape(sample_loader), loader_circuit)
                self.q_circuits[i][j] = vqc.get_circuit

                q_node = qml.QNode(vqc.get_circuit, q_device, diff_method="best", interface="torch")
                self.q_nodes[i][j] = q_node

                ml.append(qml.qnn.TorchLayer(q_node, {"var_params": var_params_shape}))
                self.param_count += sum(p.numel() for p in ml[-1].parameters())
            self.q_layers.append(ml)

        self.circuits = circuits
        self.qpc = qpc
        self.fc_in_features = len(measure_wires) * dim1_blocks * dim2_blocks
        self.fc = torch.nn.Linear(self.fc_in_features, num_classes)
        self.param_count += sum(p.numel() for p in self.fc.parameters())

    def forward(self, x):
        out_list = []
        for i in range(dim1_blocks):
            for j in range(dim2_blocks):
                block_out = self.q_layers[i][j](x[:, i * dim1_blocks + j, :])
                out_list.append(block_out)

        if postprocess:
            out = torch.stack(out_list, dim=1).reshape(-1, self.fc_in_features)
            out = self.fc(out)
        else:
            out = torch.mean(torch.stack(out_list, dim=1), dim=1)

        return out

In [ ]:
def _get_underlying(model, private):
    if private and hasattr(model, "_module"):
        return model._module, "_module."
    return model, ""


def compute_loss_hybrid(params, sample, target, model, criterion, private=False):
    batch = sample.unsqueeze(0)
    underlying, prefix = _get_underlying(model, private)

    if prefix:
        call_params = {k[len(prefix):]: v for k, v in params.items() if k.startswith(prefix)}
    else:
        call_params = params

    predictions = functional_call(underlying, call_params, (batch,))
    return criterion(predictions, target.view(1).long())


def autograd_compute_grad_hybrid(params, data_sample, target_sample, model, criterion, private=False):
    loss = compute_loss_hybrid(params, data_sample, target_sample, model, criterion, private)
    if loss.ndim != 0:
        loss = loss.mean()

    keys = list(params.keys())
    leaves = [params[k] for k in keys]
    grads = torch.autograd.grad(loss, leaves, retain_graph=False, create_graph=False)
    return dict(zip(keys, grads))


def autograd_compute_per_sample_grads_hybrid(params, X, y, model, criterion, private=False):
    grads_list = [
        autograd_compute_grad_hybrid(params, X[i], y[i], model, criterion, private)
        for i in range(X.shape[0])
    ]
    return tree_map(lambda *xs: torch.stack(xs, dim=0), *grads_list)


grad_dist = []


def train_private(net, opt, sched, n_epochs, train_loader, test_loader, private=False, privacy_config={}):
    if private:
        privacy_engine = PrivacyEngine()
        delta = privacy_config.get("target_delta", 1e-5)

        if "target_epsilon" in privacy_config:
            net, opt, train_loader = privacy_engine.make_private_with_epsilon(
                module=net,
                optimizer=opt,
                data_loader=train_loader,
                epochs=n_epochs,
                **privacy_config,
            )
        else:
            net, opt, train_loader = privacy_engine.make_private(
                module=net,
                optimizer=opt,
                data_loader=train_loader,
                **privacy_config,
            )
        print(
            f"Opacus DP-SGD enabled  |  max_grad_norm={privacy_config.get('max_grad_norm')}  "
            f"noise_multiplier={privacy_config.get('noise_multiplier')}",
            flush=True,
        )

    params = dict(net.named_parameters())
    use_no_op = private and privacy_config.get("grad_sample_mode") == "no_op"
    train_losses, test_losses, test_accs = [], [], []

    for ep in range(1, n_epochs + 1):
        net.train()
        train_loss = 0.0
        per_sample_grad_list = []

        for batch in train_loader:
            data, target = batch[0], batch[1]
            opt.zero_grad()

            output = net(data)
            loss = criterion(output, target)

            if use_no_op:
                per_sample_grads = autograd_compute_per_sample_grads_hybrid(
                    params, data, target, net, criterion, private=True
                )
                for key, value in per_sample_grads.items():
                    params[key].grad_sample = value.detach()
            else:
                loss.backward()

            if private:
                batch_per_sample_grad_list = []
                for key, param in dict(net.named_parameters()).items():
                    batch_per_sample_grad_list.append(
                        torch.reshape(param.grad_sample, (data.shape[0], -1))
                    )
                per_sample_grad_list.append(torch.cat(batch_per_sample_grad_list, dim=1))

            opt.step()
            if sched is not None:
                sched.step()

            train_loss += loss.item()

        if private:
            per_sample_grad_list = torch.cat(per_sample_grad_list, dim=0)
            grad_dist.append(per_sample_grad_list.detach())
            norms = np.sqrt(torch.sum(per_sample_grad_list ** 2, axis=1))
            print(norms.mean().item())
            print(norms.std().item())

        train_loss /= len(train_loader)
        test_loss, test_acc, _ = test(net, test_loader)
        train_losses.append(train_loss)
        test_losses.append(test_loss)
        test_accs.append(test_acc)

        if private:
            eps = privacy_engine.get_epsilon(delta)
            print(
                f"Epoch {ep:3d}  train_loss={train_loss:.4f}  "
                f"test_loss={test_loss:.4f}  f1={test_acc:.4f}  ε={eps:.3f}",
                flush=True,
            )
        else:
            print(
                f"Epoch {ep:3d}  train_loss={train_loss:.4f}  "
                f"test_loss={test_loss:.4f}  f1={test_acc:.4f}",
                flush=True,
            )

    return train_losses, test_losses, test_accs

In [ ]:
quantum_dataset = CustomLoaderDataset(
    path=loader_dir,
    class_names=class_names,
    dim1_blocks=dim1_blocks,
    dim2_blocks=dim2_blocks,
)

filter_names = ["_".join(el[-1].split("_")[-2:]) for el in quantum_dataset.data]

train_loader, test_loader = get_data_loaders(
    dataset=quantum_dataset,
    bs_train=bs_train,
    bs_test=bs_test,
    train_ratio=train_ratio,
)

In [ ]:
all_train_losses = []
all_test_losses = []
all_test_accs = []

qlayer_config_list = build_qlayer_config_list(configs)

for exp in range(n_exp):
    for config in qlayer_config_list:
        net = BAENet(num_classes=len(class_names), qlayer_config=config, noise_model=None)
        opt = optim.SGD(net.parameters(), lr=lr, momentum=momentum, weight_decay=weight_decay)

        scheduler = None
        if schedule and not private:
            scheduler = optim.lr_scheduler.OneCycleLR(
                opt, lr, epochs=n_epochs, steps_per_epoch=len(train_loader)
            )

        train_losses, test_losses, test_accs = train_private(
            net, opt, scheduler, n_epochs, train_loader, test_loader,
            private=private,
            privacy_config=privacy_config if private else {},
        )

        all_train_losses.append(train_losses)
        all_test_losses.append(test_losses)
        all_test_accs.append(test_accs)